# 🔬 Virtual Lab 4: Running OpenAI Models on LlamaIndex & LangChain  

<div style="border: 2px solid #4CAF50; padding: 15px; border-radius: 10px; background-color: #f4f4f4;">

### 🚀 **Platform**  
**OpenAI**  

### 🏷️ **Models Used**  
- **gpt-4o-mini**  
- **gpt-3.5-turbo**  

### 🛠️ **Frameworks Used**  
- **LlamaIndex**  
- **LangChain / LangGraph**  

</div>

In [1]:
!pip install pypdf2


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip show pypdf2

Name: PyPDF2
Version: 3.0.1
Summary: A pure-python PDF library capable of splitting, merging, cropping, and transforming PDF files
Home-page: 
Author: 
Author-email: Mathieu Fenniak <biziqe@mathieu.fenniak.net>
License: 
Location: C:\Workbench\LlamaIndex-LangChain-RAG\venv-LlamaIndex-LangChain\Lib\site-packages
Requires: 
Required-by: 


In [3]:
!pip install langchain langchain-core openai


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install -U langchain-community


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install -U langchain langchain-core openai langchain-community


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip install -U langchain-openai langchain langchain-core langchain-community


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import openai
import requests
import zipfile
import sqlite3
import json
from sqlalchemy import create_engine, text
from pydantic import BaseModel
from PyPDF2 import PdfReader
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

### OpenAI API Setup & Configuration

In this section, we set up the OpenAI API client and define a function (`call_gpt`)  
to interact with **GPT-4o Mini**.

In [8]:
# Set OpenAI API Key
#os.environ["OPENAI_API_KEY"] = "add-your-api-key"
#openai.api_key = os.getenv("OPENAI_API_KEY")

In [9]:
# Initialize OpenAI client
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
# Set global model configuration
llm_config = {"model": "gpt-4o-mini"}

In [11]:
# Call complete with a prompt
def call_gpt(prompt):
    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content


### Document Download & Text Extraction

This section downloads PDF documents related to **Drake and Kendrick Lamar**,  
extracts their text content, and loads them for further processing.

- **Download PDFs**: Fetches the documents from Dropbox and saves them locally.  
- **Extract Text**: Reads the PDFs using `PyPDF2` and converts them into plain text.  
- **Load Documents**: Stores the extracted text in variables (`docs_kendrick`, `docs_drake`, `docs_both`)  
  for querying and analysis.

In [12]:
# Function to download PDFs
def download_file(url, filepath):
    response = requests.get(url, stream=True)
    with open(filepath, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)

os.makedirs("data", exist_ok=True)

In [13]:
# Download documents
pdf_urls = {
    "drake_kendrick_beef": "https://www.dropbox.com/scl/fi/t1soxfjdp0v44an6sdymd/drake_kendrick_beef.pdf?rlkey=u9546ymb7fj8lk2v64r6p5r5k&st=wjzzrgil&dl=1",
    "drake": "https://www.dropbox.com/scl/fi/nts3n64s6kymner2jppd6/drake.pdf?rlkey=hksirpqwzlzqoejn55zemk6ld&st=mohyfyh4&dl=1",
    "kendrick": "https://www.dropbox.com/scl/fi/8ax2vnoebhmy44bes2n1d/kendrick.pdf?rlkey=fhxvn94t5amdqcv9vshifd3hj&st=dxdtytn6&dl=1"
}

In [14]:
for name, url in pdf_urls.items():
    download_file(url, f"data/{name}.pdf")

In [15]:
# Function to extract text from PDFs
def extract_text_from_pdf(filepath):
    with open(filepath, "rb") as file:
        reader = PdfReader(file)
        text = "\n\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
    return text

In [16]:
# Load documents
docs = {
    "drake_kendrick_beef": extract_text_from_pdf("data/drake_kendrick_beef.pdf"),
    "drake": extract_text_from_pdf("data/drake.pdf"),
    "kendrick": extract_text_from_pdf("data/kendrick.pdf")
}

In [17]:
# Initialize OpenAI Embeddings and Vector Store
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embedding=embedding_model)

In [18]:
# Add documents to vector store 
for name, text in docs.items():
    doc = Document(page_content=text, metadata={"source": name})
    vector_store.add_documents([doc])  # Remove embedding_model from here

### Basic GPT-4o Mini Query & Streaming Response

This section demonstrates how to interact with **GPT-4o Mini** using both standard  
and streaming responses.

- **Basic Completion**: Calls `call_gpt()` to get a simple text-based response.  
- **Streaming Response**: Uses `stream_gpt()` to receive output in real-time,  
  printing the response incrementally as it's generated.  

In [19]:
response = call_gpt("Do you like Drake or Kendrick better?")
print(response)

I don't have personal preferences or feelings, but both Drake and Kendrick Lamar are highly influential artists in the hip-hop genre, each with their unique style and contributions. Drake is known for his catchy hooks and versatility, while Kendrick is celebrated for his lyrical depth and storytelling. It often comes down to individual taste! Who do you prefer?


In [20]:
# Streaming response
def stream_gpt(prompt):
    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        stream=True
    )
    for chunk in response:
        if chunk.choices:
            print(chunk.choices[0].delta.content, end="")

stream_gpt("You're a Drake fan. Tell me why you like Drake more than Kendrick.")


As a fan of Drake, there are several reasons why I might prefer him over Kendrick Lamar. 

1. **Versatility**: Drake has a unique ability to blend different genres, from hip-hop and R&B to dance and pop. This versatility allows him to appeal to a broader audience and keeps his music fresh and engaging.

2. **Catchy Hooks and Melodies**: Drake is known for his memorable hooks and catchy melodies. Songs like "Hotline Bling" and "One Dance" have an infectious quality that makes them enjoyable to listen to repeatedly.

3. **Relatable Lyrics**: Drake often delves into themes of love, heartbreak, and personal struggle, which many listeners find relatable. His ability to express vulnerability in his lyrics resonates with fans who appreciate emotional depth.

4. **Consistent Output**: Drake has a prolific output, consistently releasing new music and projects. This keeps fans engaged and excited about his work, as there’s often something new to look forward to.

5. **Collaborations**: Drake fre

### Multi-Turn Chat with GPT-4o Mini

This section demonstrates **structured conversations** with GPT-4o Mini using a list of messages.

- **Role-Based Messages**: The model is assigned a **system role** (e.g., acting as Kendrick).  
- **User Interaction**: The user provides an input query, and GPT-4o Mini generates a response.    

In [21]:
# Call chat with a list of messages
def chat_with_gpt(messages):
    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content

messages = [
    {"role": "system", "content": "You are Kendrick."},
    {"role": "user", "content": "Write a verse."},
]
response = chat_with_gpt(messages)

In [22]:
print(response)

(Verse)  
Yeah, I rise from the shadows, where dreams intertwine,  
Crafting verses like a sculptor, every word a design.  
Echoes of the city, they whisper in my ear,  
Stories of the struggle, the laughter mixed with fear.  

I pen my truth in ink, let the pages bleed,  
Planting seeds of hope, in a world that’s filled with greed.  
From the block to the stage, I’m breaking every chain,  
With a heart full of fire, I’m dancing in the rain.  

So hear the rhythm of my soul, let it resonate,  
Turning pain into power, watch me elevate.  
In the tapestry of life, I’m weaving my own thread,  
With every step I take, I’m living what’s unsaid.  


### Basic RAG (Retrieval-Augmented Generation) - Vector Search

This section demonstrates **retrieving and answering questions** from documents  
using **GPT-4o Mini**.

- **Query-Based Search**: Uses `query_rag()` to fetch relevant information from the document.  
- **Contextual Responses**: The model is provided with document content to generate informed answers.   

In [23]:
def query_rag_with_embedding(prompt, top_k=3, max_tokens=3000):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)
    context = "\n\n".join([doc.page_content[:max_tokens] for doc in relevant_docs])
    full_prompt = f"Using the following retrieved information, answer the question: {prompt}\n\n{context}"

    return call_gpt(full_prompt)

In [24]:
response = query_rag_with_embedding("Tell me about family matters")
print(response)

The retrieved information primarily discusses the ongoing feud between rappers Kendrick Lamar and Drake, focusing on their recent exchanges in diss tracks and the implications of their rivalry within the hip-hop community. While it provides a detailed account of their musical careers and the dynamics of their competition, it does not specifically address "family matters."

If you are looking for insights into family matters in general, they typically refer to issues and dynamics within families, including relationships, conflicts, parenting, and support systems. Family matters can encompass legal issues such as custody disputes, inheritance, and family law, as well as emotional aspects like communication, bonding, and conflict resolution.

Please clarify if you are looking for a specific type of information regarding family matters or if you want to explore a different subject.


### Basic RAG (Retrieval-Augmented Generation) - Summarization

This section demonstrates **summarizing document content** using **GPT-4o Mini**.

- **Context-Based Summarization**: Uses `summarize_rag()` to extract key insights from documents.  
- **Efficient Information Extraction**: The model condenses long-form content into a concise response.  

In [25]:
def query_rag_with_embedding(prompt, top_k=3, max_doc_length=1000):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)    
    truncated_docs = [doc.page_content[:max_doc_length] for doc in relevant_docs]
    context = "\n\n".join(truncated_docs)
    
    full_prompt = f"Using the following retrieved information, answer the question: {prompt}\n\n{context}"
    
    response = call_gpt(full_prompt)
    
    return response


In [26]:
response = query_rag_with_embedding("Tell me about family matters")
print(response)

The retrieved information primarily details the ongoing rap feud between Kendrick Lamar and Drake, highlighting their competitive exchanges and personal diss tracks. However, it does not address "family matters" in the context of personal relationships or family dynamics. 

If you are looking for information on family matters in general, it often refers to issues related to family life, relationships, and dynamics, including parenting, marriage, divorce, and family law. It can encompass topics such as family communication, conflict resolution, and the role of family in individual development.

If you have a specific aspect of family matters you would like to explore or if you meant to ask about the family lives of Kendrick Lamar or Drake, please clarify!


### Advanced RAG (Routing & Sub-Questions)

This section implements an **intelligent query router** that determines whether  
to perform **vector search** or **summarization** based on the user's intent.

- **Automatic Routing**: GPT-4o Mini decides if a query requires **search** (fact retrieval)  
  or **summary** (document overview).  
- **Dynamic Query Processing**: The model selects the appropriate approach and generates a response.   

In [27]:
def query_router_with_embedding(prompt, top_k=3, max_tokens=3000):
    routing_prompt = (
        "Determine the best mode (search or summary) to process the given user query based on intent. "
        "Return only 'search' if the query seeks specific facts, or 'summary' if the query requires summarization. "
        "Respond with only 'search' or 'summary'.\n\n"
        f"User Query: {prompt}"
    )

    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=[{"role": "user", "content": routing_prompt}],
        temperature=0
    )

    mode = response.choices[0].message.content.strip().lower()

    relevant_docs = vector_store.similarity_search(prompt, k=top_k)
    context = "\n\n".join([doc.page_content[:max_tokens] for doc in relevant_docs]) 

    if mode == "search":
        full_prompt = f"Using the following retrieved information, find specific facts related to: {prompt}\n\n{context}"
    elif mode == "summary":
        full_prompt = f"Summarize the document with respect to: {prompt}\n\n{context}"
    else:
        full_prompt = f"{prompt}\n\n{context}" 

    return call_gpt(full_prompt)

In [28]:
response_search = query_router_with_embedding("Tell me about the song 'Meet the Grahams' - why is it significant")
print(response_search)

The song "Meet the Grahams" is significant within the context of the ongoing beef between Kendrick Lamar and Drake, two prominent figures in contemporary hip-hop. Released during a heated exchange of diss tracks, the song exemplifies the personal and competitive nature of their rivalry. Kendrick's verse in "Like That," which is part of a collaborative album with Future and Metro Boomin, marks a notable escalation in their conflict, as he directly challenges Drake's status and accomplishments in the rap game.

The significance of "Meet the Grahams" lies in its role in reshaping the dynamics of hip-hop, illustrating how personal vendettas can influence an artist's legacy and alter the rules of engagement in the genre. Kendrick's declaration of "choosing violence" and his explicit references to Drake's work signal a shift from subliminal jabs to outright confrontational lyrics, indicating that the competition is more intense and personal than ever. This song, along with others in the rece

In [29]:
response_summary = query_router_with_embedding("Summarize the significance of 'Meet the Grahams'")
print(response_summary)

The significance of "Meet the Grahams" lies primarily in its cultural context, particularly in the ongoing beef between two prominent figures in hip-hop: Kendrick Lamar and Drake. This conflict has escalated dramatically, with both artists releasing diss tracks that are not only personal but also reshape the dynamics of rap rivalries. 

The document highlights a recent surge in tension that began with Kendrick declaring "war" on Drake, resulting in a series of rapid-fire diss tracks exchanged over a single weekend. This intense back-and-forth is described as a new chapter in rap geopolitics, suggesting a shift in how artists engage in competition within the genre.

Kendrick's verses in a collaboration with Future and Metro Boomin serve as a response to Drake's perceived provocations, signaling a more confrontational approach than in their previous interactions. The significance of this beef extends beyond personal animosity; it represents a crucial moment in hip-hop history that may re

### Break Complex Questions into Sub-Questions

This section implements a **Sub-Question Query Engine** that determines  
whether a query is related to **Drake or Kendrick Lamar** and retrieves  
relevant information accordingly.

- **Automatic Subject Classification**: GPT-4o Mini classifies the query as  
  related to **Drake** or **Kendrick** before fetching data.  
- **Targeted Query Execution**: Uses `docs_drake` if the query is about Drake  
  and `docs_kendrick` if it's about Kendrick.   

In [30]:
def determine_subject(prompt):
    classification_prompt = (
        "Determine whether the following question is about 'Drake' or 'Kendrick Lamar'. "
        "Return only 'drake' or 'kendrick'.\n\n"
        f"User Question: {prompt}"
    )
    
    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=[{"role": "user", "content": classification_prompt}],
        temperature=0
    )
    
    return response.choices[0].message.content.strip().lower()

In [31]:
def sub_question_query_engine_with_embedding(prompt, top_k=3, max_tokens=3000):
    subject = determine_subject(prompt)
    
    if subject == "drake":
        relevant_docs = vector_store.similarity_search(prompt, k=top_k)
    else:  
        relevant_docs = vector_store.similarity_search(prompt, k=top_k)

    context = "\n\n".join([doc.page_content[:max_tokens] for doc in relevant_docs])

    full_prompt = f"Using the following retrieved information, answer the question: {prompt}\n\n{context}"

    return call_gpt(full_prompt)


In [32]:
response = sub_question_query_engine_with_embedding("Which albums did Drake release in his career?")
print(response)

Drake has released the following albums throughout his career:

1. **Thank Me Later** (2010)
2. **Take Care** (2011)
3. **Nothing Was the Same** (2013)
4. **Views** (2016)
5. **Scorpion** (2018)
6. **Certified Lover Boy** (2021)
7. **Honestly, Nevermind** (2022)
8. **Her Loss** (2022) - Collaborative album with 21 Savage
9. **For All the Dogs** (2023)

In addition to these studio albums, Drake has also released several mixtapes, including his debut **Room for Improvement** (2006), **Comeback Season** (2007), and **So Far Gone** (2009).


### Text-to-SQL with GPT-4o Mini

This section demonstrates **converting natural language queries into SQL**  
to retrieve data from an SQLite database.

- **Database Setup**:  
  - Downloads and extracts the **Chinook SQLite database**, which contains  
    music-related tables like `albums`, `artists`, and `tracks`.  
  - Initializes a connection to `chinook.db` using SQLAlchemy.  

- **SQL Query Generation**:  
  - Uses GPT-4o Mini to **convert natural language questions into SQL queries**.  
  - Restricts queries to the tables: `albums`, `artists`, and `tracks`.  

- **Query Execution**:  
  - Runs the generated SQL queries on the database and retrieves the results.  

This setup allows **seamless querying of structured data** using natural language. 🚀

In [33]:
def download_file(url, filepath):
    response = requests.get(url, stream=True)
    with open(filepath, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)

# Create data directory
os.makedirs("data", exist_ok=True)

In [34]:
download_file("https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip", "data/chinook.zip")
with zipfile.ZipFile("data/chinook.zip", "r") as zip_ref:
    zip_ref.extractall("data/")

In [35]:
engine = create_engine("sqlite:///data/chinook.db")

In [36]:
tables_schema = {
    "albums": "AlbumId, Title, ArtistId",
    "artists": "ArtistId, Name",
    "tracks": "TrackId, Name, AlbumId, Composer, MediaTypeId, GenreId, Milliseconds, Bytes, UnitPrice"
}

In [37]:
schema_docs = [
    Document(page_content=f"Table: {table}\nColumns: {columns}", metadata={"table": table})
    for table, columns in tables_schema.items()
]

In [38]:
vector_store.add_documents(schema_docs)

['64b8e0f9-5416-46f4-b02c-4deb759d94b1',
 'bfe3529d-943b-439e-9fb8-b757205afd11',
 '0e704dcf-ec7d-4112-a1a9-9a73e2cee2ed']

In [39]:
def retrieve_relevant_schema(prompt, top_k=2):
    relevant_schema = vector_store.similarity_search(prompt, k=top_k)
    return "\n\n".join([doc.page_content for doc in relevant_schema])

In [40]:
def generate_sql_query_with_embedding(natural_language_query):
    relevant_schema = retrieve_relevant_schema(natural_language_query)

    prompt = f"""
    Convert the following natural language question into a SQL query for a SQLite database.
    Use only the relevant schema details provided below:

    {relevant_schema}

    **Now generate a SQL query for the following request:**
    
    "{natural_language_query}"
    
    **Return only the SQL query, without any explanation.**
    """

    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    sql_query = response.choices[0].message.content.strip()

    if "```sql" in sql_query:
        sql_query = sql_query.split("```sql")[1].split("```")[0].strip()

    return sql_query

In [41]:
from sqlalchemy.sql import text  

def query_sql_database_with_embedding(natural_language_query):
    sql_query = generate_sql_query_with_embedding(natural_language_query)
    print(f"Generated SQL Query: {sql_query}")
    with engine.connect() as connection:
        result = connection.execute(text(sql_query))  
        return [row for row in result]

In [42]:
response_albums = query_sql_database_with_embedding("What are some albums?")
print("Albums:", response_albums)

Generated SQL Query: SELECT * FROM albums;
Albums: [(1, 'For Those About To Rock We Salute You', 1), (2, 'Balls to the Wall', 2), (3, 'Restless and Wild', 2), (4, 'Let There Be Rock', 1), (5, 'Big Ones', 3), (6, 'Jagged Little Pill', 4), (7, 'Facelift', 5), (8, 'Warner 25 Anos', 6), (9, 'Plays Metallica By Four Cellos', 7), (10, 'Audioslave', 8), (11, 'Out Of Exile', 8), (12, 'BackBeat Soundtrack', 9), (13, 'The Best Of Billy Cobham', 10), (14, 'Alcohol Fueled Brewtality Live! [Disc 1]', 11), (15, 'Alcohol Fueled Brewtality Live! [Disc 2]', 11), (16, 'Black Sabbath', 12), (17, 'Black Sabbath Vol. 4 (Remaster)', 12), (18, 'Body Count', 13), (19, 'Chemical Wedding', 14), (20, 'The Best Of Buddy Guy - The Millenium Collection', 15), (21, 'Prenda Minha', 16), (22, 'Sozinho Remix Ao Vivo', 16), (23, 'Minha Historia', 17), (24, 'Afrociberdelia', 18), (25, 'Da Lama Ao Caos', 18), (26, 'Acústico MTV [Live]', 19), (27, 'Cidade Negra - Hits', 19), (28, 'Na Pista', 20), (29, 'Axé Bahia 2001', 21)

In [43]:
response_artists = query_sql_database_with_embedding("What are some artists? Limit it to 5.")
print("Artists:", response_artists)

Generated SQL Query: SELECT Name FROM artists LIMIT 5;
Artists: [('AC/DC',), ('Accept',), ('Aerosmith',), ('Alanis Morissette',), ('Alice In Chains',)]


In [44]:
response_tracks = query_sql_database_with_embedding("What are some tracks from the artist AC/DC? Limit it to 3")
print("AC/DC Tracks:", response_tracks)


Generated SQL Query: SELECT Name FROM tracks WHERE Composer = 'AC/DC' LIMIT 3;
AC/DC Tracks: [('Go Down',), ('Dog Eat Dog',), ('Let There Be Rock',)]


### Structured Data Extraction using GPT-4o Mini

This section demonstrates **extracting structured data** from natural language  
using **GPT-4o Mini** and returning it in **JSON format**.

- **Data Extraction Process**:  
  - The model generates structured data for a **restaurant** in a given city.  
  - The output must be a **valid JSON object** containing:  
    - `name`: The restaurant's name.  
    - `city`: The specified city.  
    - `cuisine`: The type of cuisine served.  

In [45]:
from langchain_core.documents import Document
from pydantic import BaseModel

class Restaurant(BaseModel):
    """A restaurant with name, city, and cuisine."""
    name: str
    city: str
    cuisine: str

In [46]:
restaurant_data = [
    {"name": "Joe's Seafood", "city": "Miami", "cuisine": "Seafood"},
    {"name": "Pasta Paradise", "city": "New York", "cuisine": "Italian"},
    {"name": "Sushi Haven", "city": "San Francisco", "cuisine": "Japanese"},
    {"name": "BBQ King", "city": "Austin", "cuisine": "BBQ"},
]

In [47]:
restaurant_docs = [
    Document(page_content=f"Restaurant: {r['name']}, City: {r['city']}, Cuisine: {r['cuisine']}", metadata={"city": r["city"]})
    for r in restaurant_data
]

vector_store.add_documents(restaurant_docs)


['2256919c-d3a7-42b7-ac5c-4a30fb43d08f',
 'a2a48f7b-aecd-4031-9a88-0eda4141d1b4',
 '5f962cb6-e957-4a2b-b4a0-6b58dd2e1744',
 '0e483fe7-fe83-44a5-8072-20eef98fdfdf']

In [48]:
def retrieve_restaurants_by_city(city_name, top_k=3):
    relevant_docs = vector_store.similarity_search(city_name, k=top_k)
    return relevant_docs

In [49]:
def extract_structured_data_with_embedding(city_name):
    relevant_restaurants = retrieve_restaurants_by_city(city_name)

    if not relevant_restaurants:
        prompt = f"Generate a restaurant in a given city: {city_name}. Return only a valid JSON object with keys: name, city, cuisine, without markdown formatting."
        response = client.chat.completions.create(
            model=llm_config["model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        structured_data = response.choices[0].message.content.strip()

        if "```json" in structured_data:
            structured_data = structured_data.split("```json")[1].split("```")[0].strip()

        try:
            json_data = json.loads(structured_data)
            return Restaurant.model_validate(json_data)
        except json.JSONDecodeError as e:
            print("JSON Decode Error:", e)
            print("Raw Response:", structured_data)
            return None

    restaurant_list = []
    for doc in relevant_restaurants:
        parts = doc.page_content.split(", ")
        restaurant_dict = {
            "name": parts[0].split(": ")[1],
            "city": parts[1].split(": ")[1],
            "cuisine": parts[2].split(": ")[1],
        }
        restaurant_list.append(Restaurant(**restaurant_dict))

    return restaurant_list[0] if restaurant_list else None  # Return the first matching restaurant

In [50]:
restaurant_obj = extract_structured_data_with_embedding("Miami")
print(restaurant_obj)

name="Joe's Seafood" city='Miami' cuisine='Seafood'


### Adding Chat History to RAG (Chat Engine)

This section implements a **stateful chatbot** that integrates **chat history**  
with **Retrieval-Augmented Generation (RAG)** to provide more context-aware responses.

- **Chat Memory Management**:  
  - Stores past interactions in `ChatMemory` to maintain conversation flow.  
  - Limits stored messages to prevent exceeding token constraints.  

- **Contextual Retrieval**:  
  - Combines **user input, past chat history, and relevant document context**  
    (e.g., about Kendrick & Drake) to generate informed responses.  

In [51]:
from langchain_core.documents import Document

class ChatMemoryWithEmbeddings:
    def __init__(self, token_limit=10000): 
        self.token_limit = token_limit
        self.messages = []
        self.vector_store = vector_store  

    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})

        doc = Document(page_content=content[:1000], metadata={"role": role}) 
        self.vector_store.add_documents([doc])

        if len(self.messages) > 20:  
            self.messages.pop(0)

    def retrieve_relevant_history(self, prompt, top_k=2):
        relevant_docs = self.vector_store.similarity_search(prompt, k=top_k)
        return "\n\n".join([doc.page_content[:500] for doc in relevant_docs])  


In [52]:
memory = ChatMemoryWithEmbeddings()

In [53]:
def chat_with_history_using_embeddings(prompt):
    relevant_history = memory.retrieve_relevant_history(prompt, top_k=2)

    relevant_docs = vector_store.similarity_search(prompt, k=2)
    document_context = "\n\n".join([doc.page_content[:1000] for doc in relevant_docs])  # Limit document size

    context_prompt = (
        "You are a chatbot, able to have normal interactions, as well as talk "
        "about the Kendrick and Drake beef. Use the retrieved chat history and document context:\n\n"
        f"Chat History (trimmed):\n{relevant_history}\n\n"
        f"Relevant Documents (trimmed):\n{document_context}\n\n"
        "Instruction: Use the previous chat history, or the context above, to interact and help the user."
    )

    messages = [
        {"role": "system", "content": context_prompt},
        {"role": "user", "content": prompt}
    ]

    total_tokens = sum(len(msg["content"].split()) for msg in messages)
    if total_tokens > 10000:
        print(f"Warning: Trimming messages to fit within token limit.")
        messages = messages[-5:]  

    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=messages,
        temperature=0.7
    )

    response_content = response.choices[0].message.content.strip()
    memory.add_message("assistant", response_content) 

    return response_content



In [54]:
response = chat_with_history_using_embeddings("Tell me about the songs Drake released in the beef.")
print(response)

During the recent escalation of the beef between Drake and Kendrick Lamar, Drake released a three-part response that garnered significant attention. This release included diss tracks that were densely packed with personal jabs aimed at Kendrick. The accompanying music video also added a visual element to the confrontation, amplifying the impact of his words.

The specific titles of the songs in this three-part series weren't mentioned in the context provided, but Drake's approach typically involves clever wordplay and references that highlight his dominance in the rap game. His ability to craft catchy hooks while delivering sharp bars is part of what makes his responses so engaging.

Kendrick Lamar quickly followed up with his own response, showcasing the competitive nature of their rivalry. This back-and-forth has kept fans on the edge of their seats, eagerly anticipating each artist's next move. If you're interested in deep dives into specific lyrics or themes from these tracks, let 

In [55]:
response = chat_with_history_using_embeddings("What about Kendrick?")
print(response)

Kendrick Lamar is known for his sharp lyricism and profound social commentary in his music. He often addresses themes of race, identity, and injustice, which has resonated deeply with listeners. In the context of his rivalry with Drake, Kendrick has also shown his competitive edge through clever wordplay and powerful diss tracks. 

During their recent beef, Kendrick responded to Drake's three-part diss series with his own tracks, demonstrating his lyrical prowess and willingness to engage in the battle. This ongoing tension between the two has kept fans buzzing, as each artist brings their unique style and perspective to the table. If you're curious about specific tracks or lyrics from Kendrick in this rivalry, feel free to ask!


## 7. Agents

Here we build agents with gpt-4o-mini . We perform RAG over simple functions as well as the documents above.

In [56]:
import nest_asyncio
import json
from langchain_core.documents import Document

nest_asyncio.apply()

In [57]:
# Define mathematical functions
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b

def subtract(a: int, b: int) -> int:
    """Subtract two integers and return the result."""
    return a - b

def divide(a: int, b: int) -> int:
    """Divide two integers and return the result."""
    return a / b if b != 0 else "Cannot divide by zero"

# Function map for the agent
tools = {
    "multiply": multiply,
    "add": add,
    "subtract": subtract,
    "divide": divide
}

In [58]:
function_examples = [
    {"function": "multiply", "query": "What is 5 times 3?", "args": [5, 3]},
    {"function": "add", "query": "What is 10 plus 4?", "args": [10, 4]},
    {"function": "subtract", "query": "What is 20 minus 7?", "args": [20, 7]},
    {"function": "divide", "query": "What is 15 divided by 5?", "args": [15, 5]},
]

In [59]:
function_docs = [
    Document(page_content=f"Query: {ex['query']}, Function: {ex['function']}, Args: {ex['args']}")
    for ex in function_examples
]

vector_store.add_documents(function_docs)

['d44af7f8-c93c-4a99-ba49-4e0e1c5c8f1d',
 '7149ca44-f511-4ec0-a4a8-75ca601171a9',
 'f2995959-a0cf-46e7-a643-b18c4fdef41d',
 '766f0513-4bb7-410b-862f-4d5a65cbc676']

In [60]:
def retrieve_relevant_function_examples(prompt, top_k=2):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)
    return "\n\n".join([doc.page_content for doc in relevant_docs])


In [61]:
def execute_tool(tool_name, *args):
    if tool_name in tools:
        return tools[tool_name](*args)
    return "Invalid tool request"


In [62]:
def agent_chat_with_embeddings(prompt):
    relevant_examples = retrieve_relevant_function_examples(prompt)

    system_prompt = (
        "You are a smart assistant capable of performing arithmetic operations. "
        "You can use the following functions: multiply, add, subtract, divide. "
        "Below are relevant past function calls for reference:\n\n"
        f"{relevant_examples}\n\n"
        "When given a math question, return the correct function and inputs in JSON format. "
        "Ensure the JSON output follows this format:\n\n"
        "{\n  \"function\": \"function_name\",\n  \"arguments\": [arg1, arg2]\n}\n\n"
        "Return only valid JSON with no extra text or formatting."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=messages,
        temperature=0
    )

    response_content = response.choices[0].message.content.strip()

    try:
        if "```json" in response_content:
            response_content = response_content.split("```json")[1].split("```")[0].strip()

        tool_request = json.loads(response_content)
        tool_name = tool_request.get("function")
        arguments = tool_request.get("arguments", [])

        result = execute_tool(tool_name, *arguments)
        return result
    except json.JSONDecodeError:
        print("Failed to parse response:", response_content)
        return "Error: Could not parse the response as JSON."


In [63]:
response = agent_chat_with_embeddings("What is (121 + 2) * 5?")
print(response)

615


### ReAct Agent with RAG QueryEngine Tools

This section implements a **ReAct-style agent** that dynamically selects between  
**Drake-related** and **Kendrick-related** document retrieval using **GPT-4o Mini**.

- **ReAct Framework**:  
  - Uses **reasoning + action** to **select the right tool** for querying.  
  - Determines whether to call **`query_drake`** or **`query_kendrick`** based on the prompt.  

- **Tool-Based Query Execution**:  
  - **GPT-4o Mini** analyzes the query and responds in **JSON format** specifying the correct tool.  
  - The tool is then **executed dynamically** to fetch relevant information.  

This approach enables **intelligent document selection** and **enhanced retrieval accuracy**. 🚀

In [64]:
import json
from langchain_core.documents import Document

def query_rag_with_embedding(prompt, top_k=3, max_tokens=3000):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)

    context = "\n\n".join([doc.page_content[:max_tokens] for doc in relevant_docs])

    full_prompt = f"Using the following retrieved information, answer the question: {prompt}\n\n{context}"

    return call_gpt(full_prompt)

In [65]:
def determine_subject_using_embeddings(prompt):
    classification_prompt = (
        "Determine whether the following question is about 'Drake' or 'Kendrick Lamar'. "
        "Return only 'drake' or 'kendrick'.\n\n"
        f"User Query: {prompt}"
    )
    
    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=[{"role": "user", "content": classification_prompt}],
        temperature=0
    )
    
    return response.choices[0].message.content.strip().lower()

In [66]:
def react_agent_with_embeddings(prompt):
    subject = determine_subject_using_embeddings(prompt)

    relevant_docs = vector_store.similarity_search(prompt, k=3)
    document_context = "\n\n".join([doc.page_content[:1000] for doc in relevant_docs])  # Limit document size

    system_prompt = (
        f"You are an AI assistant capable of retrieving and summarizing information about Drake and Kendrick Lamar. "
        f"Use the retrieved document context below to generate your response:\n\n"
        f"Relevant Documents (trimmed):\n{document_context}\n\n"
        f"Instruction: Answer the user's query using the provided context."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model=llm_config["model"],
        messages=messages,
        temperature=0
    )

    return response.choices[0].message.content.strip()


In [67]:
response = react_agent_with_embeddings("Tell me about how Kendrick and Drake grew up")
print(response)

The provided context does not include specific details about Kendrick Lamar and Drake's upbringing. However, I can provide a general overview based on what is commonly known.

Kendrick Lamar grew up in Compton, California, an area known for its struggles with crime and poverty. His experiences in this environment heavily influenced his music, as he often addresses themes of race, identity, and social injustice in his lyrics.

Drake, on the other hand, was raised in Toronto, Canada. He had a mixed background, with a Jewish mother and an African American father. Drake's early life was marked by his experiences in the entertainment industry, as he started acting on the television show "Degrassi: The Next Generation" before transitioning to music.

Both artists have drawn from their unique backgrounds to shape their musical identities, contributing to their rivalry and the themes they explore in their work. If you would like more specific information or details about their childhoods, feel